# 12 — MCP 协议：接入外部工具服务

前几章实现的工具（ReadFileTool、WebSearchTool 等）都是**硬编码在代码里的**——加工具得改代码，重新部署。

MCP（Model Context Protocol）解决这个问题：用统一协议让 AI 动态发现并调用外部工具服务，不需要修改 Agent 代码。

---

**类比 USB**：
- 以前：每种设备专用接口，换设备要换线（硬编码工具）
- 现在：USB 统一接口，插什么设备都能用（MCP 协议）

---

这章实现：
1. MCP 概念和三种传输方式
2. `MCPToolWrapper`：把 MCP 工具包装成 Tool 接口
3. `MCPClient`：连接 MCP 服务器，发现和调用工具
4. `MCPManager`：管理多个 MCP 连接
5. `config.toml` 中的 MCP 配置格式
6. 写一个简单的 MCP Server 并连接测试

## 1. MCP 架构与传输方式

MCP 是 Anthropic 2024 年发布的开放标准协议，定义了 AI 客户端和工具服务器之间的通信格式。

```
┌─────────────────────────────────────────────────────────────┐
│                        AI Agent                             │
│   ContextManager + ToolRegistry + AgenticLoop               │
└──────────────────────┬──────────────────────────────────────┘
                       │ 通过 MCP 协议
          ┌────────────┼────────────┐
          ▼            ▼            ▼
   ┌──────────┐ ┌──────────┐ ┌──────────┐
   │ STDIO    │ │ HTTP     │ │ SSE      │
   │ 本地进程  │ │ 远程服务  │ │ 流式推送  │
   └──────────┘ └──────────┘ └──────────┘
        │              │            │
   本地 MCP      远程 MCP       远程 MCP
   Server        Server         Server
```

### 三种传输方式

| 传输方式 | 场景 | 特点 |
|----------|------|------|
| **STDIO** | 本地进程（最常用） | 启动子进程，stdin/stdout 通信，低延迟 |
| **HTTP** | 远程服务 | 标准 REST 风格，无状态，易部署 |
| **SSE** | 服务器推送 | 服务器主动推送事件，适合长时任务 |

### MCP 的核心操作

```
客户端发起                    服务器响应
─────────────────────────────────────────
initialize()          →  server info + capabilities
list_tools()          →  [{ name, description, inputSchema }]
call_tool(name, args) →  { content: [...] }
```

`inputSchema` 就是 JSON Schema，和 OpenAI function calling 的格式一样——这正是可以无缝包装的原因。

In [ ]:
# 检查依赖
import subprocess, sys

def check_package(pkg_name: str, import_name: str = None) -> bool:
    """检查包是否可以导入"""
    if import_name is None:
        import_name = pkg_name
    try:
        __import__(import_name)
        return True
    except ImportError:
        return False

HAS_MCP = check_package("mcp")

print(f"mcp 库: {'✓ 已安装' if HAS_MCP else '✗ 未安装 (pip install mcp)'}")
print()
if not HAS_MCP:
    print("安装命令: pip install mcp")
    print("本章核心代码在两种情况下都能运行：")
    print("  - 有 mcp 库：使用真实 STDIO 连接")
    print("  - 无 mcp 库：使用 HTTP 降级方案")

## 2. MCPToolWrapper：把 MCP 工具包装成 Tool 接口

MCP server 返回的工具信息格式：

```json
{
  "name": "run_tests",
  "description": "运行 pytest 测试",
  "inputSchema": {
    "type": "object",
    "properties": {
      "test_path": { "type": "string", "description": "测试路径" }
    },
    "required": ["test_path"]
  }
}
```

`MCPToolWrapper` 把这个 dict 和一个调用函数包装成 `Tool` 对象，让 `ToolRegistry` 完全透明地管理 MCP 工具——注册进去之后和本地工具完全一样。

In [ ]:
import sys
sys.path.insert(0, "..")

from typing import Callable, Awaitable
from src.tool_framework import Tool, ToolKind, ToolResult


class MCPToolWrapper(Tool):
    """
    把 MCP server 返回的工具信息包装成 Tool 接口。

    这样 MCP 工具和本地工具可以统一注册到 ToolRegistry，
    Agent 代码不需要区分工具来自哪里。
    """

    def __init__(
        self,
        mcp_tool_info: dict,
        call_fn: Callable[[str, dict], Awaitable[str]],
    ):
        """
        mcp_tool_info : MCP server 返回的工具描述 dict
                        必须包含 name, description, inputSchema
        call_fn       : async (name, arguments) -> str
                        实际调用 MCP server 的函数
        """
        self._info = mcp_tool_info
        self._call_fn = call_fn

    # ── Tool 接口实现 ──────────────────────────────────────────

    @property
    def name(self) -> str:
        return self._info["name"]

    @property
    def description(self) -> str:
        return self._info.get("description", "")

    @property
    def kind(self) -> ToolKind:
        # MCP 工具统一标记为 MCP 类型
        return ToolKind.MCP

    def schema(self) -> dict:
        """
        把 MCP 的 inputSchema 转换成 OpenAI function calling 格式。
        MCP inputSchema 本身就是 JSON Schema，直接嵌入即可。
        """
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self._info.get("inputSchema", {
                    "type": "object",
                    "properties": {},
                }),
            },
        }

    def validate(self, params: dict) -> list[str]:
        """基于 inputSchema 的 required 字段做基本验证"""
        errors = []
        schema = self._info.get("inputSchema", {})
        required = schema.get("required", [])
        for field in required:
            if field not in params:
                errors.append(f"缺少必填参数: {field}")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        """调用 MCP server 执行工具"""
        try:
            content = await self._call_fn(self.name, params)
            return ToolResult.ok(content, mcp_tool=self.name)
        except Exception as e:
            return ToolResult.error(f"MCP 工具调用失败: {e}", mcp_tool=self.name)


# ── 单元测试 ──────────────────────────────────────────────────
import asyncio

# 模拟 MCP server 返回的工具信息
fake_tool_info = {
    "name": "run_tests",
    "description": "运行 pytest 测试，返回测试结果",
    "inputSchema": {
        "type": "object",
        "properties": {
            "test_path": {
                "type": "string",
                "description": "测试路径，默认为当前目录",
            }
        },
        "required": [],
    },
}

# 模拟调用函数（不真正连接 MCP server）
async def fake_call_fn(name: str, arguments: dict) -> str:
    return f"PASSED 3 tests in {arguments.get('test_path', '.')} (mocked)"

# 创建包装器
wrapper = MCPToolWrapper(fake_tool_info, fake_call_fn)
print(repr(wrapper))
print(f"kind: {wrapper.kind}")
print()

# 检查 schema 转换
import json
print("schema():")
print(json.dumps(wrapper.schema(), indent=2, ensure_ascii=False))

In [ ]:
# 执行测试
result = await wrapper.execute({"test_path": "tests/"})
print(f"success  : {result.success}")
print(f"content  : {result.content}")
print(f"metadata : {result.metadata}")
print()

# 验证必填参数（这个工具没有必填，所以通过）
print("validate 空参数:", wrapper.validate({}))

# 测试有必填参数的工具
strict_tool_info = {
    "name": "strict_tool",
    "description": "有必填参数的工具",
    "inputSchema": {
        "type": "object",
        "properties": {
            "path": {"type": "string"},
            "content": {"type": "string"},
        },
        "required": ["path", "content"],
    },
}
strict_wrapper = MCPToolWrapper(strict_tool_info, fake_call_fn)
print("validate 缺少必填参数:", strict_wrapper.validate({"path": "/tmp/x"}))
print("validate 参数齐全:", strict_wrapper.validate({"path": "/tmp/x", "content": "hello"}))

## 3. MCPClient：连接 MCP 服务器

`MCPClient` 负责：
1. 启动 MCP server 子进程（STDIO 模式）
2. 建立连接，调用 `list_tools()` 获取工具列表
3. 把每个工具包装成 `MCPToolWrapper`
4. 提供 `call_tool()` 方法供 wrapper 回调
5. 维护连接状态

```
MCPClient.connect()
    │
    ├─ 启动子进程: npx @modelcontextprotocol/server-filesystem /tmp
    │
    ├─ stdio_client() 建立 STDIO 连接
    │
    ├─ ClientSession.initialize()
    │
    ├─ session.list_tools()  →  [ToolInfo, ...]
    │
    └─ 每个 ToolInfo → MCPToolWrapper(info, self.call_tool)
```

In [ ]:
from enum import Enum
from typing import Optional
import asyncio
import logging

logger = logging.getLogger(__name__)


class MCPConnectionState(Enum):
    """MCP 连接状态机"""
    DISCONNECTED = "disconnected"
    CONNECTING   = "connecting"
    CONNECTED    = "connected"
    ERROR        = "error"


class MCPClient:
    """
    MCP 客户端（STDIO 模式）。

    连接一个 MCP server 进程，发现其提供的工具，包装成 MCPToolWrapper。
    """

    def __init__(
        self,
        name: str,
        command: str,
        args: list[str] = None,
        env: dict[str, str] = None,
    ):
        """
        name    : 这个 MCP server 的标识名（用于日志/状态报告）
        command : 启动命令，例如 "npx" 或 "python"
        args    : 命令参数列表
        env     : 追加的环境变量
        """
        self.name = name
        self.command = command
        self.args = args or []
        self.env = env or {}

        self._state = MCPConnectionState.DISCONNECTED
        self._tools: list[MCPToolWrapper] = []
        self._session = None
        self._exit_stack = None
        self._error: Optional[Exception] = None

    @property
    def state(self) -> MCPConnectionState:
        return self._state

    @property
    def tools(self) -> list[MCPToolWrapper]:
        return self._tools

    # ── 连接 ──────────────────────────────────────────────────

    async def connect(self) -> list[MCPToolWrapper]:
        """
        启动 MCP server 进程，建立 STDIO 连接，返回工具列表。
        连接失败时抛出异常，调用方负责捕获。
        """
        if not HAS_MCP:
            raise RuntimeError("mcp 库未安装，无法使用 STDIO 连接。请运行: pip install mcp")

        from mcp import ClientSession, StdioServerParameters
        from mcp.client.stdio import stdio_client
        from contextlib import AsyncExitStack
        import os

        self._state = MCPConnectionState.CONNECTING
        logger.info(f"[{self.name}] 正在连接: {self.command} {' '.join(self.args)}")

        try:
            # 合并环境变量
            merged_env = {**os.environ, **self.env} if self.env else None

            server_params = StdioServerParameters(
                command=self.command,
                args=self.args,
                env=merged_env,
            )

            # AsyncExitStack 管理 stdio_client 的生命周期
            self._exit_stack = AsyncExitStack()
            stdio = await self._exit_stack.enter_async_context(
                stdio_client(server_params)
            )
            read_stream, write_stream = stdio

            # 建立 MCP 会话
            self._session = await self._exit_stack.enter_async_context(
                ClientSession(read_stream, write_stream)
            )
            await self._session.initialize()

            # 获取工具列表
            tools_response = await self._session.list_tools()
            self._tools = [
                MCPToolWrapper(
                    mcp_tool_info={
                        "name": t.name,
                        "description": t.description or "",
                        "inputSchema": t.inputSchema if hasattr(t, "inputSchema") else {},
                    },
                    call_fn=self.call_tool,
                )
                for t in tools_response.tools
            ]

            self._state = MCPConnectionState.CONNECTED
            logger.info(f"[{self.name}] 已连接，发现 {len(self._tools)} 个工具: "
                        f"{[t.name for t in self._tools]}")
            return self._tools

        except Exception as e:
            self._state = MCPConnectionState.ERROR
            self._error = e
            logger.error(f"[{self.name}] 连接失败: {e}")
            if self._exit_stack:
                await self._exit_stack.aclose()
                self._exit_stack = None
            raise

    # ── 调用工具 ──────────────────────────────────────────────

    async def call_tool(self, name: str, arguments: dict) -> str:
        """
        调用 MCP server 上的工具。
        这个方法作为 call_fn 传给 MCPToolWrapper。
        """
        if self._state != MCPConnectionState.CONNECTED or self._session is None:
            raise RuntimeError(f"[{self.name}] 未连接，无法调用工具 '{name}'")

        result = await self._session.call_tool(name, arguments)

        # 提取文本内容（MCP 返回的是结构化内容列表）
        parts = []
        for item in result.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            else:
                parts.append(str(item))
        return "\n".join(parts)

    # ── 断开连接 ──────────────────────────────────────────────

    async def disconnect(self):
        """断开 MCP server 连接，释放资源"""
        if self._exit_stack:
            await self._exit_stack.aclose()
            self._exit_stack = None
        self._session = None
        self._state = MCPConnectionState.DISCONNECTED
        self._tools = []
        logger.info(f"[{self.name}] 已断开")

    def get_status(self) -> dict:
        """返回连接状态信息"""
        return {
            "name": self.name,
            "command": f"{self.command} {' '.join(self.args)}",
            "state": self._state.value,
            "tool_count": len(self._tools),
            "tools": [t.name for t in self._tools],
            "error": str(self._error) if self._error else None,
        }

    def __repr__(self):
        return f"<MCPClient {self.name} [{self._state.value}] {len(self._tools)} tools>"


print("MCPClient 定义完成")
print()

# 展示状态机
for s in MCPConnectionState:
    print(f"  {s.value}")

## 4. MCPManager：管理多个 MCP 连接

实际使用时，一个 Agent 可能同时连接多个 MCP server（文件系统、GitHub、数据库……）。

`MCPManager` 负责：
- 并发初始化所有 MCP 连接（`asyncio.gather`）
- 把发现的所有工具注册到 `ToolRegistry`
- 优雅关闭所有连接
- 提供统一的状态报告

In [ ]:
from src.tool_framework import ToolRegistry


class MCPManager:
    """
    管理多个 MCP 客户端连接。

    初始化时并发连接所有 MCP server，
    把它们提供的工具统一注册到 ToolRegistry。
    """

    def __init__(self, clients: list[MCPClient], registry: ToolRegistry):
        """
        clients  : MCPClient 列表，每个对应一个 MCP server
        registry : 工具注册表，MCP 工具会注册进来
        """
        self._clients = clients
        self._registry = registry
        self._initialized = False

    async def initialize(self) -> dict[str, list[str]]:
        """
        并发连接所有 MCP server，把工具注册到 registry。
        返回每个 server 发现的工具名列表。
        连接失败的 server 会记录错误但不中断整体初始化。
        """
        results: dict[str, list[str]] = {}

        # 并发连接，单个失败不影响其他
        async def connect_one(client: MCPClient):
            try:
                tools = await client.connect()
                for tool in tools:
                    self._registry.register(tool)
                results[client.name] = [t.name for t in tools]
                print(f"  [{client.name}] 已连接，注册 {len(tools)} 个工具")
            except Exception as e:
                results[client.name] = []
                print(f"  [{client.name}] 连接失败: {e}")

        await asyncio.gather(*[connect_one(c) for c in self._clients])
        self._initialized = True
        return results

    async def shutdown(self):
        """并发断开所有 MCP 连接"""
        await asyncio.gather(
            *[c.disconnect() for c in self._clients],
            return_exceptions=True,  # 单个断开失败不中断整体
        )
        self._initialized = False
        print("所有 MCP 连接已关闭")

    def get_status(self) -> list[dict]:
        """返回所有 client 的状态列表"""
        return [c.get_status() for c in self._clients]

    def __repr__(self):
        connected = sum(1 for c in self._clients if c.state == MCPConnectionState.CONNECTED)
        return f"<MCPManager {connected}/{len(self._clients)} connected>"


print("MCPManager 定义完成")

## 5. config.toml 中的 MCP 配置

把 MCP server 配置写在 `config.toml` 里，Agent 启动时读取并初始化。

### 配置格式

```toml
[mcp.servers.filesystem]
enabled = true
command = "npx"
args    = ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]

[mcp.servers.github]
enabled = false
command = "npx"
args    = ["-y", "@modelcontextprotocol/server-github"]

[mcp.servers.demo]
enabled = true
command = "python"
args    = ["course/demo_mcp_server.py"]
```

**字段说明**：
- `enabled`：false 的 server 不会被初始化（方便临时禁用）
- `command`：启动命令，`npx` 用于 npm 包，`python` 用于本地脚本
- `args`：命令行参数，第一个通常是 MCP server 包名或脚本路径

### 从配置创建 MCPClient

In [ ]:
import tomllib
from pathlib import Path


def load_mcp_clients_from_config(config_path: str = "../config.toml") -> list[MCPClient]:
    """
    从 config.toml 读取 MCP 配置，返回 MCPClient 列表。
    只返回 enabled = true 的 server。
    config.toml 不存在时返回空列表。
    """
    path = Path(config_path)
    if not path.exists():
        print(f"config.toml 不存在: {path.resolve()}，跳过 MCP 配置")
        return []

    with open(path, "rb") as f:
        config = tomllib.load(f)

    servers = config.get("mcp", {}).get("servers", {})
    clients = []
    for name, cfg in servers.items():
        if not cfg.get("enabled", True):
            print(f"  [{name}] 已禁用，跳过")
            continue
        client = MCPClient(
            name=name,
            command=cfg["command"],
            args=cfg.get("args", []),
            env=cfg.get("env", {}),
        )
        clients.append(client)
        print(f"  [{name}] 已加载: {cfg['command']} {' '.join(cfg.get('args', []))}")

    return clients


# 演示（config.toml 不存在也没关系）
print("从 config.toml 加载 MCP 配置:")
clients = load_mcp_clients_from_config("../config.toml")
print(f"加载到 {len(clients)} 个 MCP client")

In [ ]:
# 演示 config.toml 的内容（写一个示例文件展示格式）
config_example = """[mcp.servers.filesystem]
enabled = true
command = "npx"
args    = ["-y", "@modelcontextprotocol/server-filesystem", "/tmp"]

[mcp.servers.github]
enabled = false
command = "npx"
args    = ["-y", "@modelcontextprotocol/server-github"]

[mcp.servers.demo]
enabled = true
command = "python"
args    = ["course/demo_mcp_server.py"]
"""

print("config.toml 示例格式:")
print("-" * 50)
print(config_example)

# 用 tomllib 解析验证格式正确
import tomllib
parsed = tomllib.loads(config_example)
servers = parsed["mcp"]["servers"]
for name, cfg in servers.items():
    enabled = cfg.get("enabled", True)
    status = "enabled" if enabled else "disabled"
    print(f"  {name:15s} [{status:8s}] {cfg['command']} {' '.join(cfg.get('args', []))}")

## 6. 写一个 MCP Server（演示用）

用 `mcp` 库写一个最简单的本地 MCP server。

**要求**：`mcp` 库内置了 `FastMCP` 框架，用装饰器语法注册工具，和 FastAPI 很像。

这个 server 提供一个 `run_tests` 工具，实际调用 pytest 运行测试。

In [ ]:
# 把 MCP server 代码写到文件
demo_server_code = '''"""
demo_mcp_server.py — 演示用 MCP server

用法（STDIO 模式，由 MCPClient 自动启动）:
    python demo_mcp_server.py

或者直接测试:
    python demo_mcp_server.py --test
"""
import sys

try:
    from mcp.server.fastmcp import FastMCP
except ImportError:
    print("需要安装 mcp 库: pip install mcp", file=sys.stderr)
    sys.exit(1)

app = FastMCP("demo")


@app.tool()
def run_tests(test_path: str = ".") -> str:
    """运行 pytest 测试，返回测试结果"""
    import subprocess
    result = subprocess.run(
        ["python", "-m", "pytest", test_path, "-v", "--tb=short"],
        capture_output=True,
        text=True,
        timeout=60,
    )
    output = result.stdout + result.stderr
    return output if output.strip() else "没有测试输出"


@app.tool()
def list_python_files(directory: str = ".") -> str:
    """列出目录中的所有 Python 文件"""
    import os
    py_files = []
    for root, dirs, files in os.walk(directory):
        # 跳过隐藏目录和 __pycache__
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
        for f in files:
            if f.endswith(".py"):
                rel_path = os.path.relpath(os.path.join(root, f), directory)
                py_files.append(rel_path)
    if not py_files:
        return f"目录 {directory} 中没有 Python 文件"
    return "\\n".join(sorted(py_files))


@app.tool()
def get_system_info() -> str:
    """获取系统基本信息（Python 版本、平台等）"""
    import platform
    return (
        f"Python: {sys.version}\\n"
        f"Platform: {platform.platform()}\\n"
        f"CWD: {__import__(\"os\").getcwd()}"
    )


if __name__ == "__main__":
    if "--test" in sys.argv:
        # 简单的手动测试模式（不通过 MCP 协议，直接调函数）
        print("=== 直接调用测试 ===")
        print("get_system_info():")
        print(get_system_info())
        print()
        print("list_python_files('.'):")
        print(list_python_files("."))
    else:
        # 正常 MCP STDIO 模式
        app.run()
'''

server_path = "../course/demo_mcp_server.py"
with open(server_path, "w") as f:
    f.write(demo_server_code)

print(f"已写入: {server_path}")
print()
print("提供的工具:")
print("  - run_tests(test_path)    : 运行 pytest 测试")
print("  - list_python_files(dir)  : 列出 Python 文件")
print("  - get_system_info()       : 获取系统信息")

In [ ]:
# 验证 server 可以直接运行（--test 模式，不走 MCP 协议）
import subprocess, sys

result = subprocess.run(
    [sys.executable, "../course/demo_mcp_server.py", "--test"],
    capture_output=True,
    text=True,
    timeout=10,
)

if result.returncode == 0:
    print("demo_mcp_server.py 运行正常:")
    print(result.stdout)
else:
    print("运行失败:")
    print(result.stderr)

## 7. 连接 Demo Server 并调用工具

用 `MCPClient` 连接刚才写的 demo server，完整走一遍 MCP 连接流程：

```
MCPClient → 启动子进程 demo_mcp_server.py
         → STDIO 握手（initialize）
         → list_tools() → 3 个工具
         → MCPToolWrapper × 3
         → ToolRegistry.register()
         → invoke("get_system_info", {})
```

In [ ]:
import asyncio
import sys
import os

async def test_mcp_connection():
    if not HAS_MCP:
        print("mcp 库未安装，跳过连接测试")
        print("安装: pip install mcp")
        return

    # 用绝对路径确保子进程能找到脚本
    server_script = os.path.abspath("../course/demo_mcp_server.py")
    if not os.path.exists(server_script):
        print(f"Server 脚本不存在: {server_script}")
        return

    client = MCPClient(
        name="demo",
        command=sys.executable,   # 当前 Python 解释器
        args=[server_script],
    )

    registry = ToolRegistry()

    print("=== MCPClient 连接测试 ===")
    print(f"启动命令: {sys.executable} {server_script}")
    print()

    try:
        # 连接并发现工具
        tools = await client.connect()
        print(f"发现 {len(tools)} 个工具:")
        for t in tools:
            print(f"  - {t.name}: {t.description[:50]}")
            registry.register(t)
        print()

        # 调用 get_system_info
        print("调用 get_system_info():")
        result = await registry.invoke("get_system_info", {})
        print(f"success: {result.success}")
        print(f"content:\n{result.content}")
        print()

        # 调用 list_python_files
        print("调用 list_python_files('..'):")
        result2 = await registry.invoke("list_python_files", {"directory": ".."})
        print(f"success: {result2.success}")
        # 只打印前 5 行
        lines = result2.content.split("\n")[:5]
        print("\n".join(lines))
        if result2.content.count("\n") > 5:
            print(f"  ... (共 {result2.content.count(chr(10))+1} 个文件)")

    except Exception as e:
        print(f"连接或调用失败: {type(e).__name__}: {e}")
    finally:
        await client.disconnect()
        print("\n连接已关闭")
        print(f"最终状态: {client.state.value}")


await test_mcp_connection()

In [ ]:
# 测试 MCPManager：管理多个连接（含失败处理）
async def test_mcp_manager():
    if not HAS_MCP:
        print("mcp 库未安装，跳过 MCPManager 测试")
        return

    server_script = os.path.abspath("../course/demo_mcp_server.py")

    # 两个 client：一个真实的，一个不存在的命令（测试失败处理）
    clients = [
        MCPClient("demo", sys.executable, [server_script]),
        MCPClient("nonexistent", "nonexistent_command_xyz", ["--flag"]),
    ]

    registry = ToolRegistry()
    manager = MCPManager(clients, registry)

    print("=== MCPManager 初始化测试 ===")
    print("正在并发连接（一个成功，一个预期失败）...")
    print()

    await manager.initialize()
    print()
    print(f"ToolRegistry 已注册工具: {[t.name for t in registry.list_tools()]}")
    print()

    # 状态报告
    print("状态报告:")
    for status in manager.get_status():
        print(f"  [{status['name']}] {status['state']:12s} | "
              f"工具数: {status['tool_count']} | "
              f"{status['tools']}")
        if status['error']:
            print(f"    错误: {status['error'][:80]}")

    await manager.shutdown()


await test_mcp_manager()

## 8. 降级方案：HTTP MCP Client

如果 `mcp` 库没有安装，或者需要连接 HTTP 模式的 MCP server，可以用 `httpx` 直接调 HTTP API。

HTTP MCP server 的接口：

```
POST /tools/list      → [{ name, description, inputSchema }]
POST /tools/call      → { content: [{type: "text", text: "..."}] }
  body: { name: "tool_name", arguments: {...} }
```

In [ ]:
import httpx


class HTTPMCPClient:
    """
    HTTP 模式 MCP 客户端（降级方案）。

    当 mcp 库不可用，或 MCP server 以 HTTP 方式运行时使用。
    协议：POST /tools/list 获取工具列表，POST /tools/call 调用工具。
    """

    def __init__(self, name: str, base_url: str, timeout: float = 30.0):
        """
        name     : client 标识名
        base_url : HTTP MCP server 地址，如 http://localhost:8080
        timeout  : 请求超时（秒）
        """
        self.name = name
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

        self._state = MCPConnectionState.DISCONNECTED
        self._tools: list[MCPToolWrapper] = []
        self._error: Exception | None = None

    @property
    def state(self) -> MCPConnectionState:
        return self._state

    async def connect(self) -> list[MCPToolWrapper]:
        """发起 list_tools 请求，获取工具列表"""
        self._state = MCPConnectionState.CONNECTING
        try:
            async with httpx.AsyncClient(timeout=self.timeout) as http:
                resp = await http.post(f"{self.base_url}/tools/list", json={})
                resp.raise_for_status()
                data = resp.json()

            tools_data = data.get("tools", data) if isinstance(data, dict) else data
            self._tools = [
                MCPToolWrapper(
                    mcp_tool_info=t,
                    call_fn=self.call_tool,
                )
                for t in tools_data
            ]
            self._state = MCPConnectionState.CONNECTED
            return self._tools

        except Exception as e:
            self._state = MCPConnectionState.ERROR
            self._error = e
            raise

    async def call_tool(self, name: str, arguments: dict) -> str:
        """调用 MCP server 上的工具"""
        async with httpx.AsyncClient(timeout=self.timeout) as http:
            resp = await http.post(
                f"{self.base_url}/tools/call",
                json={"name": name, "arguments": arguments},
            )
            resp.raise_for_status()
            result = resp.json()

        # 提取文本内容
        content = result.get("content", [])
        parts = [
            item["text"] for item in content
            if isinstance(item, dict) and item.get("type") == "text"
        ]
        return "\n".join(parts) if parts else str(result)

    async def disconnect(self):
        """HTTP 模式无需特殊关闭操作"""
        self._state = MCPConnectionState.DISCONNECTED

    def get_status(self) -> dict:
        return {
            "name": self.name,
            "command": f"HTTP {self.base_url}",
            "state": self._state.value,
            "tool_count": len(self._tools),
            "tools": [t.name for t in self._tools],
            "error": str(self._error) if self._error else None,
        }


print("HTTPMCPClient 定义完成")
print()
print("使用方式（HTTP MCP server 在 localhost:8080 时）:")
print("  client = HTTPMCPClient('my-server', 'http://localhost:8080')")
print("  tools = await client.connect()")
print("  result = await client.call_tool('run_tests', {'test_path': './'})")

In [ ]:
# 演示 HTTP client 的错误处理（server 不存在时）
async def test_http_client_error():
    client = HTTPMCPClient("test-server", "http://localhost:19999")
    print("测试 HTTP 连接失败处理:")
    try:
        await client.connect()
    except Exception as e:
        print(f"  连接失败（预期）: {type(e).__name__}")
        print(f"  状态: {client.state.value}")
        print(f"  get_status(): {client.get_status()}")

await test_http_client_error()

## 9. 完整集成测试：MCP 工具 + Agent 对话

把 MCP 工具接入完整的 Agent 对话流程：

1. 启动 demo MCP server
2. 发现工具并注册到 ToolRegistry
3. 用户提问 → LLM 决定调用 MCP 工具 → 执行 → 返回结果

In [ ]:
import json
from src.llm_client import LLMClient
from src.context_manager import ContextManager


async def run_mcp_agent_demo():
    if not HAS_MCP:
        print("mcp 库未安装，使用模拟演示")
        await run_mock_demo()
        return

    server_script = os.path.abspath("../course/demo_mcp_server.py")

    # 初始化 MCP client 和 registry
    client = MCPClient("demo", sys.executable, [server_script])
    registry = ToolRegistry()

    print("=== MCP + Agent 集成演示 ===")

    try:
        tools = await client.connect()
        for t in tools:
            registry.register(t)
        print(f"已注册 {len(tools)} 个 MCP 工具: {[t.name for t in tools]}")
        print()

        # 初始化 LLM 和对话上下文
        llm = LLMClient()
        ctx = ContextManager(
            system_prompt="你是一个 Python 开发助手，可以用工具查看系统信息和文件。"
        )

        # 用户问题
        user_question = "用工具查一下系统信息"
        ctx.add_user_message(user_question)
        print(f"用户: {user_question}")
        print()

        # 调用 LLM（带工具列表）
        response_text, usage = await llm.chat_completion(
            messages=ctx.get_messages(),
            tools=registry.get_schemas(),
        )

        print(f"LLM 回复类型: {type(response_text).__name__}")

        # 检查是否有工具调用（LLM 可能直接回答或调工具）
        from openai import AsyncOpenAI
        raw = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

        raw_resp = await raw.chat.completions.create(
            model="gpt-oss:120b",
            messages=ctx.get_messages(),
            tools=registry.get_schemas(),
        )
        msg = raw_resp.choices[0].message

        if msg.tool_calls:
            ctx.add_assistant_message(
                content=msg.content or "",
                tool_calls=[tc.model_dump() for tc in msg.tool_calls],
            )
            for tc in msg.tool_calls:
                name = tc.function.name
                params = json.loads(tc.function.arguments)
                print(f"LLM 调用工具: {name}({params})")

                result = await registry.invoke(name, params)
                print(f"工具结果 (前 200 字): {result.content[:200]}")
                print()
                ctx.add_tool_result(tc.id, result.content)

            # 最终回复
            final_resp = await raw.chat.completions.create(
                model="gpt-oss:120b",
                messages=ctx.get_messages(),
            )
            print("LLM 最终回复:")
            print(final_resp.choices[0].message.content)
        else:
            print("LLM 直接回复（未调工具）:")
            print(msg.content)

        await raw.close()
        await llm.close()

    except Exception as e:
        print(f"演示过程出错: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
    finally:
        await client.disconnect()


async def run_mock_demo():
    """mcp 库不存在时的模拟演示"""
    print("=== 模拟演示（无真实 MCP 连接）===")

    # 用 fake_call_fn 模拟工具执行
    tool_infos = [
        {
            "name": "get_system_info",
            "description": "获取系统基本信息",
            "inputSchema": {"type": "object", "properties": {}},
        },
        {
            "name": "list_python_files",
            "description": "列出目录中的 Python 文件",
            "inputSchema": {
                "type": "object",
                "properties": {"directory": {"type": "string"}},
            },
        },
    ]

    async def mock_call(name: str, arguments: dict) -> str:
        if name == "get_system_info":
            return "Python: 3.11.0\nPlatform: macOS\nCWD: /tmp"
        elif name == "list_python_files":
            return "agent.py\nmain.py\nsrc/llm_client.py"
        return "工具调用成功"

    registry = ToolRegistry()
    for info in tool_infos:
        registry.register(MCPToolWrapper(info, mock_call))

    print(f"已注册 {len(registry)} 个模拟工具: {[t.name for t in registry.list_tools()]}")
    print()

    result = await registry.invoke("get_system_info", {})
    print(f"get_system_info() 结果:\n{result.content}")
    print()

    result2 = await registry.invoke("list_python_files", {"directory": "."})
    print(f"list_python_files('.') 结果:\n{result2.content}")


await run_mcp_agent_demo()

## 10. 错误场景处理

MCP 连接在实际使用中有几种常见失败模式，必须正确处理：

| 场景 | 原因 | 处理方式 |
|------|------|----------|
| server 启动失败 | 命令不存在/权限不足 | connect() 抛异常，MCPManager 记录错误继续初始化 |
| server 崩溃 | 子进程意外退出 | call_tool() 抛异常，MCPToolWrapper.execute() 包装成 ToolResult.error |
| 工具参数错误 | 调用方传了错误类型 | validate() 提前返回错误，不发给 server |
| 超时 | server 处理太慢 | asyncio.timeout 包裹 call_tool |
| 重连 | 断线后需要恢复 | 重新调用 connect()，state 从 DISCONNECTED 开始 |

In [ ]:
# 演示各种错误场景

async def test_error_scenarios():
    print("=== MCP 错误场景测试 ===")
    print()

    # 场景 1：server 命令不存在
    print("场景 1: server 命令不存在")
    if HAS_MCP:
        bad_client = MCPClient("bad", "nonexistent_cmd_xyz", [])
        try:
            await bad_client.connect()
        except Exception as e:
            print(f"  connect() 抛出: {type(e).__name__}")
            print(f"  client 状态: {bad_client.state.value}")
    else:
        print("  （mcp 未安装，跳过）")
    print()

    # 场景 2：工具参数验证失败
    print("场景 2: 必填参数缺失")
    strict_info = {
        "name": "write_file",
        "description": "写文件",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
        },
    }
    async def dummy_call(name, args): return "ok"
    tool = MCPToolWrapper(strict_info, dummy_call)
    errors = tool.validate({"path": "/tmp/x"})  # 缺少 content
    print(f"  validate 结果: {errors}")
    print()

    # 场景 3：ToolRegistry 中工具调用时的错误捕获
    print("场景 3: 工具执行时抛出异常")
    async def failing_call(name, args):
        raise ConnectionError("MCP server 连接中断")

    crash_tool = MCPToolWrapper(
        {"name": "crash", "description": "会崩", "inputSchema": {"type": "object", "properties": {}}},
        failing_call,
    )
    # MCPToolWrapper.execute() 会捕获异常并返回 ToolResult.error
    result = await crash_tool.execute({})
    print(f"  is_error: {result.is_error}")
    print(f"  content: {result.content}")
    print()

    # 场景 4：通过 ToolRegistry 调用（双层保护）
    print("场景 4: 通过 ToolRegistry 调用崩溃工具")
    reg = ToolRegistry()
    reg.register(crash_tool)
    result2 = await reg.invoke("crash", {})
    print(f"  is_error: {result2.is_error}")
    print(f"  content: {result2.content}")
    print()

    # 场景 5：调用未注册工具
    print("场景 5: 调用未注册工具")
    result3 = await reg.invoke("unknown_tool", {})
    print(f"  is_error: {result3.is_error}")
    print(f"  content: {result3.content}")


await test_error_scenarios()

In [ ]:
# 演示带超时的工具调用
import asyncio


async def call_tool_with_timeout(
    registry: ToolRegistry,
    tool_name: str,
    params: dict,
    timeout_sec: float = 5.0,
) -> ToolResult:
    """带超时限制的工具调用包装"""
    try:
        async with asyncio.timeout(timeout_sec):
            return await registry.invoke(tool_name, params)
    except asyncio.TimeoutError:
        return ToolResult.error(
            f"工具 '{tool_name}' 执行超时（>{timeout_sec}s）",
            tool=tool_name,
            timeout=timeout_sec,
        )


# 测试超时处理
async def slow_call(name, args):
    await asyncio.sleep(10)  # 模拟慢 server
    return "never reached"

slow_tool = MCPToolWrapper(
    {"name": "slow_tool", "description": "很慢的工具",
     "inputSchema": {"type": "object", "properties": {}}},
    slow_call,
)

reg_timeout = ToolRegistry()
reg_timeout.register(slow_tool)

print("测试超时处理（超时设为 0.1s）:")
result = await call_tool_with_timeout(reg_timeout, "slow_tool", {}, timeout_sec=0.1)
print(f"is_error : {result.is_error}")
print(f"content  : {result.content}")

## 11. 保存到 src/

In [ ]:
import os
os.makedirs("../src", exist_ok=True)

mcp_code = '''"""
src/mcp_client.py — MCP 协议客户端

提供：
  MCPConnectionState  - 连接状态枚举
  MCPToolWrapper      - 把 MCP 工具包装成 Tool 接口
  MCPClient           - STDIO 模式 MCP 客户端
  HTTPMCPClient       - HTTP 模式 MCP 客户端（降级方案）
  MCPManager          - 管理多个 MCP 连接
  load_mcp_clients_from_config - 从 config.toml 加载配置
"""
import asyncio
import logging
import tomllib
from enum import Enum
from pathlib import Path
from typing import Callable, Awaitable, Optional

import httpx

from .tool_framework import Tool, ToolKind, ToolResult, ToolRegistry

logger = logging.getLogger(__name__)

try:
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
    HAS_MCP = True
except ImportError:
    HAS_MCP = False


class MCPConnectionState(Enum):
    DISCONNECTED = "disconnected"
    CONNECTING   = "connecting"
    CONNECTED    = "connected"
    ERROR        = "error"


class MCPToolWrapper(Tool):
    """
    把 MCP server 返回的工具信息包装成 Tool 接口。
    MCP 工具和本地工具可以统一注册到 ToolRegistry。
    """

    def __init__(
        self,
        mcp_tool_info: dict,
        call_fn: Callable[[str, dict], Awaitable[str]],
    ):
        self._info = mcp_tool_info
        self._call_fn = call_fn

    @property
    def name(self) -> str:
        return self._info["name"]

    @property
    def description(self) -> str:
        return self._info.get("description", "")

    @property
    def kind(self) -> ToolKind:
        return ToolKind.MCP

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self._info.get("inputSchema", {
                    "type": "object",
                    "properties": {},
                }),
            },
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        required = self._info.get("inputSchema", {}).get("required", [])
        for field in required:
            if field not in params:
                errors.append(f"缺少必填参数: {field}")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        try:
            content = await self._call_fn(self.name, params)
            return ToolResult.ok(content, mcp_tool=self.name)
        except Exception as e:
            return ToolResult.error(f"MCP 工具调用失败: {e}", mcp_tool=self.name)


class MCPClient:
    """STDIO 模式 MCP 客户端"""

    def __init__(
        self,
        name: str,
        command: str,
        args: list[str] = None,
        env: dict[str, str] = None,
    ):
        self.name = name
        self.command = command
        self.args = args or []
        self.env = env or {}
        self._state = MCPConnectionState.DISCONNECTED
        self._tools: list[MCPToolWrapper] = []
        self._session = None
        self._exit_stack = None
        self._error: Optional[Exception] = None

    @property
    def state(self) -> MCPConnectionState:
        return self._state

    @property
    def tools(self) -> list[MCPToolWrapper]:
        return self._tools

    async def connect(self) -> list[MCPToolWrapper]:
        if not HAS_MCP:
            raise RuntimeError("mcp 库未安装，请运行: pip install mcp")

        from contextlib import AsyncExitStack
        import os

        self._state = MCPConnectionState.CONNECTING
        try:
            merged_env = {**os.environ, **self.env} if self.env else None
            server_params = StdioServerParameters(
                command=self.command,
                args=self.args,
                env=merged_env,
            )
            self._exit_stack = AsyncExitStack()
            stdio = await self._exit_stack.enter_async_context(
                stdio_client(server_params)
            )
            read_stream, write_stream = stdio
            self._session = await self._exit_stack.enter_async_context(
                ClientSession(read_stream, write_stream)
            )
            await self._session.initialize()

            tools_response = await self._session.list_tools()
            self._tools = [
                MCPToolWrapper(
                    mcp_tool_info={
                        "name": t.name,
                        "description": t.description or "",
                        "inputSchema": t.inputSchema if hasattr(t, "inputSchema") else {},
                    },
                    call_fn=self.call_tool,
                )
                for t in tools_response.tools
            ]
            self._state = MCPConnectionState.CONNECTED
            logger.info(f"[{self.name}] 已连接，发现 {len(self._tools)} 个工具")
            return self._tools

        except Exception as e:
            self._state = MCPConnectionState.ERROR
            self._error = e
            if self._exit_stack:
                await self._exit_stack.aclose()
                self._exit_stack = None
            raise

    async def call_tool(self, name: str, arguments: dict) -> str:
        if self._state != MCPConnectionState.CONNECTED or self._session is None:
            raise RuntimeError(f"[{self.name}] 未连接")
        result = await self._session.call_tool(name, arguments)
        parts = []
        for item in result.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            else:
                parts.append(str(item))
        return "\n".join(parts)

    async def disconnect(self):
        if self._exit_stack:
            await self._exit_stack.aclose()
            self._exit_stack = None
        self._session = None
        self._state = MCPConnectionState.DISCONNECTED
        self._tools = []

    def get_status(self) -> dict:
        return {
            "name": self.name,
            "command": f"{self.command} {\' \'.join(self.args)}",
            "state": self._state.value,
            "tool_count": len(self._tools),
            "tools": [t.name for t in self._tools],
            "error": str(self._error) if self._error else None,
        }

    def __repr__(self):
        return f"<MCPClient {self.name} [{self._state.value}] {len(self._tools)} tools>"


class HTTPMCPClient:
    """HTTP 模式 MCP 客户端（降级方案）"""

    def __init__(self, name: str, base_url: str, timeout: float = 30.0):
        self.name = name
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self._state = MCPConnectionState.DISCONNECTED
        self._tools: list[MCPToolWrapper] = []
        self._error: Exception | None = None

    @property
    def state(self) -> MCPConnectionState:
        return self._state

    async def connect(self) -> list[MCPToolWrapper]:
        self._state = MCPConnectionState.CONNECTING
        try:
            async with httpx.AsyncClient(timeout=self.timeout) as http:
                resp = await http.post(f"{self.base_url}/tools/list", json={})
                resp.raise_for_status()
                data = resp.json()
            tools_data = data.get("tools", data) if isinstance(data, dict) else data
            self._tools = [
                MCPToolWrapper(mcp_tool_info=t, call_fn=self.call_tool)
                for t in tools_data
            ]
            self._state = MCPConnectionState.CONNECTED
            return self._tools
        except Exception as e:
            self._state = MCPConnectionState.ERROR
            self._error = e
            raise

    async def call_tool(self, name: str, arguments: dict) -> str:
        async with httpx.AsyncClient(timeout=self.timeout) as http:
            resp = await http.post(
                f"{self.base_url}/tools/call",
                json={"name": name, "arguments": arguments},
            )
            resp.raise_for_status()
            result = resp.json()
        content = result.get("content", [])
        parts = [
            item["text"] for item in content
            if isinstance(item, dict) and item.get("type") == "text"
        ]
        return "\n".join(parts) if parts else str(result)

    async def disconnect(self):
        self._state = MCPConnectionState.DISCONNECTED

    def get_status(self) -> dict:
        return {
            "name": self.name,
            "command": f"HTTP {self.base_url}",
            "state": self._state.value,
            "tool_count": len(self._tools),
            "tools": [t.name for t in self._tools],
            "error": str(self._error) if self._error else None,
        }


class MCPManager:
    """管理多个 MCP 客户端连接"""

    def __init__(self, clients: list[MCPClient | HTTPMCPClient], registry: ToolRegistry):
        self._clients = clients
        self._registry = registry
        self._initialized = False

    async def initialize(self) -> dict[str, list[str]]:
        results: dict[str, list[str]] = {}

        async def connect_one(client):
            try:
                tools = await client.connect()
                for tool in tools:
                    self._registry.register(tool)
                results[client.name] = [t.name for t in tools]
            except Exception as e:
                results[client.name] = []
                logger.error(f"[{client.name}] 连接失败: {e}")

        await asyncio.gather(*[connect_one(c) for c in self._clients])
        self._initialized = True
        return results

    async def shutdown(self):
        await asyncio.gather(
            *[c.disconnect() for c in self._clients],
            return_exceptions=True,
        )
        self._initialized = False

    def get_status(self) -> list[dict]:
        return [c.get_status() for c in self._clients]

    def __repr__(self):
        connected = sum(
            1 for c in self._clients
            if c.state == MCPConnectionState.CONNECTED
        )
        return f"<MCPManager {connected}/{len(self._clients)} connected>"


def load_mcp_clients_from_config(
    config_path: str = "config.toml",
) -> list[MCPClient]:
    """从 config.toml 的 [mcp.servers.*] 段加载 MCPClient 列表"""
    path = Path(config_path)
    if not path.exists():
        return []
    with open(path, "rb") as f:
        config = tomllib.load(f)
    servers = config.get("mcp", {}).get("servers", {})
    clients = []
    for name, cfg in servers.items():
        if not cfg.get("enabled", True):
            continue
        clients.append(MCPClient(
            name=name,
            command=cfg["command"],
            args=cfg.get("args", []),
            env=cfg.get("env", {}),
        ))
    return clients
'''

# 写入文件（去掉开头结尾的引号缩进）
with open("../src/mcp_client.py", "w") as f:
    f.write(mcp_code.strip())

print("已保存: src/mcp_client.py")
print()
print("导入方式:")
print("  from src.mcp_client import (")
print("      MCPConnectionState, MCPToolWrapper,")
print("      MCPClient, HTTPMCPClient, MCPManager,")
print("      load_mcp_clients_from_config,")
print("  )")

## 小结

| 模块 | 作用 |
|------|------|
| `MCPConnectionState` | 连接状态枚举：DISCONNECTED / CONNECTING / CONNECTED / ERROR |
| `MCPToolWrapper` | 把 MCP server 的工具信息包装成 `Tool` 接口，与本地工具统一管理 |
| `MCPClient` | STDIO 模式客户端，启动本地子进程，通过标准输入输出通信 |
| `HTTPMCPClient` | HTTP 模式客户端，连接远程 MCP server，mcp 库不可用时的降级方案 |
| `MCPManager` | 并发管理多个 MCP 连接，统一注册到 ToolRegistry，单个失败不中断整体 |
| `load_mcp_clients_from_config` | 从 `config.toml` 的 `[mcp.servers.*]` 段读取配置，返回 MCPClient 列表 |
| `demo_mcp_server.py` | 演示用 MCP server，提供 run_tests / list_python_files / get_system_info |

**设计要点**：`MCPToolWrapper` 实现了 `Tool` 接口，意味着 MCP 工具和第 03-05 章的本地工具对 Agent 完全透明——`ToolRegistry` 不知道也不关心某个工具是本地实现还是来自外部 MCP server。

下一章实现 **Agent 主循环**：把 LLMClient、ContextManager、ToolRegistry（含 MCP 工具）串联成完整的多轮对话+工具调用循环，加上循环终止条件、最大轮次限制和流式输出支持。